# Avaliacao de Rotas Medicas

Este notebook demonstra apenas a base de avaliacao da rota: carregar os pontos, calcular distancias, decodificar um cromossomo manual e calcular a fitness.

O cromossomo usado aqui e fixo apenas para teste da funcao `evaluate()`.

Configuracao inicial para importar os modulos do projeto.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DEFAULT_CONFIG
from src.data_loader import get_hospitals, get_origin, get_supply_stations, load_points
from src.distance import build_distance_matrix
from src.fitness import evaluate

## 1. Carregar catalogo de pontos

O catalogo contem a origem, os hospitais obrigatorios e os pontos de abastecimento.

In [ ]:
points = load_points(PROJECT_ROOT / "data" / "pontos_entrega.csv")

origin = get_origin(points)
hospitals = get_hospitals(points)
supply_stations = get_supply_stations(points)

print(f"Origem: {origin.idx} - {origin.name}")
print(f"Hospitais: {len(hospitals)}")
print(f"Abastecimentos: {len(supply_stations)}")
print(f"Total de pontos: {len(points)}")

for point in points:
    print(f"{point.idx:>3} | {point.type:<8} | {point.name:<24} | prioridade={point.priority or '-':<5} | demanda={point.demand}")

## 2. Calcular matriz de distancias

A matriz guarda a distancia entre cada par de pontos. Ela nao gera rotas nem testa combinacoes; apenas evita recalcular distancias.

In [ ]:
distance_matrix = build_distance_matrix(points)

print(f"Pares calculados: {len(distance_matrix)}")
print(f"Exemplo 0 -> 1: {distance_matrix[(0, 1)]:.4f}")
print(f"Exemplo 1 -> 0: {distance_matrix[(1, 0)]:.4f}")

## 3. Avaliar um cromossomo manual

Nesta etapa, o cromossomo e informado manualmente apenas para demonstrar a funcao de avaliacao. A capacidade inicial do veiculo vem de `DEFAULT_CONFIG.vehicle_capacity`.

In [ ]:
chromosome = [3, 1, 5, 2, 4]

result = evaluate(chromosome, points, distance_matrix, DEFAULT_CONFIG)

print(f"Capacidade inicial do veiculo: {DEFAULT_CONFIG.vehicle_capacity}")
print(f"Peso da prioridade: {DEFAULT_CONFIG.lambda_priority}")
print(f"Peso do abastecimento: {DEFAULT_CONFIG.lambda_supply}")
print()
print(f"Cromossomo: {result.chromosome}")
print(f"Rota decodificada: {result.decoded_route}")
print(f"Distancia total: {result.total_distance:.4f}")
print(f"Penalidade de prioridade: {result.priority_penalty:.2f}")
print(f"Penalidade de abastecimento: {result.supply_penalty:.2f}")
print(f"Reabastecimentos: {result.resupply_count}")
print(f"Fitness: {result.fitness:.2f}")
print(f"Valido: {result.is_valid}")
print(f"Erros: {result.errors}")

## 4. Exemplo com abastecimento

Neste exemplo, a capacidade do veiculo e 100. As demandas dos hospitais escolhidos sao 40, 30 e 35. Depois das duas primeiras entregas, restam 30 de carga; como o proximo hospital demanda 35, o decoder insere um abastecimento antes da entrega.

In [ ]:
chromosome_with_resupply = [7, 10, 5]
result_with_resupply = evaluate(chromosome_with_resupply, points, distance_matrix, DEFAULT_CONFIG)

for hospital_idx in chromosome_with_resupply:
    hospital = next(point for point in points if point.idx == hospital_idx)
    print(f"Hospital {hospital.idx}: demanda={hospital.demand}")

print()
print(f"Carga inicial: {DEFAULT_CONFIG.vehicle_capacity}")
print(f"Cromossomo: {result_with_resupply.chromosome}")
print(f"Rota decodificada: {result_with_resupply.decoded_route}")
print(f"Reabastecimentos: {result_with_resupply.resupply_count}")
print(f"Penalidade de abastecimento: {result_with_resupply.supply_penalty:.2f}")
print(f"Fitness: {result_with_resupply.fitness:.2f}")

## 5. Uso esperado

A funcao principal entregue e:

```python
result = evaluate(chromosome, points, distance_matrix, config)
```

Ela recebe um cromossomo pronto e devolve a rota decodificada, a distancia, as penalidades e a fitness.